# 05 — Results Visualization

Training curves and example captions for one run. Edit `CONFIG` / `SEED`.

In [ ]:
import sys, json, os
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
CONFIG, SEED = 'hybrid_frozen', 1
RUN = f'../runs/{CONFIG}/seed{SEED}'

## Training curves

In [ ]:
h = json.load(open(f'{RUN}/history.json'))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(h['loss'], label='train'); axes[0].plot(h['val_loss'], label='val'); axes[0].set_title('Loss (padding ignored)')
for k in ['masked_acc', 'top5_acc']:
    axes[1].plot(h[f'val_{k}'], label=f'val {k}')
axes[1].set_title('Validation accuracy'); [a.legend() or a.set_xlabel('epoch') for a in axes]
plt.tight_layout(); plt.savefig(f'../outputs/curves_{CONFIG}.png', dpi=150); plt.show()

## Best and worst captions (beam 5)

In [ ]:
p = pd.read_csv(f'{RUN}/eval/predictions.csv')
p = p[p.decoding == 'beam5'].sort_values('bleu4')
IMG_DIR = '/content/data_local/raw/Images'   # Colab path from notebook 00
def show(rows, title):
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4)); fig.suptitle(title)
    for ax, (_, r) in zip(axes, rows.iterrows()):
        if os.path.exists(f'{IMG_DIR}/{r.image}'): ax.imshow(Image.open(f'{IMG_DIR}/{r.image}'))
        ax.set_title(r.caption, fontsize=9, wrap=True); ax.axis('off')
    plt.tight_layout(); plt.show()
show(p.tail(4), 'Best (highest BLEU-4)')
show(p.head(4), 'Worst (lowest BLEU-4)')